In [ ]:
!pip install ultralytics deep_sort_realtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 21.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# -*- coding: utf-8 -*-
"""
여러 영상 일괄 처리 — 단일 영상 로직 100% 유지 + GPU 가속
- 모델은 1회 로드(GPU/FP16/torch.compile 옵션)
- 각 영상마다 트래커/상태 초기화
- 첫 번째 코드의 휴리스틱/파라미터/시각화 완전 동일
"""

import os
import cv2
import math
import numpy as np
from collections import deque

import torch
from ultralytics import RTDETR
from deep_sort_realtime.deepsort_tracker import DeepSort

# =========================================================
# 런타임 / 장치 설정 (GPU 자동 사용)
# =========================================================
try:
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass
except Exception:
    pass

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
USE_HALF = (DEVICE.startswith("cuda"))
IMG_SIZE = None            # 예: 960 지정 가능. None이면 모델 기본값 사용
USE_TORCH_COMPILE = True   # 환경에 따라 자동 실패-무시

# =========================================================
# 설정부 (첫 코드와 동일)
# =========================================================
YOLO_WEIGHTS = '/content/best_detr.pt'  # ✅ 첫 번째 코드와 동일 경로 사용

TRASH_LABELS = ['trash']
CAR_LABEL = 'car'

VIDEO_DIR = '/content/drive/MyDrive/video_ours'   # ✅ 폴더 경로
OUTPUT_ROOT = 'littering_clips_943_ours'                 # ✅ 출력 루트

COLORS = {
    'car': (255, 0, 0),
    'littering_car': (0, 0, 255),
    'trash': (255, 0, 255),
    'text': (220, 220, 220),
    'window_strip': (180, 180, 0),
    'trash_traj': (200, 120, 255),
    'human': (0, 255, 255),
    'hand': (255, 255, 255),
    'license_plate': (0, 255, 0)
}

# =========================================================
# 파라미터 (첫 코드 값 유지)
# =========================================================
CONF_THRES = 0.6
WINDOW_STRIP_RATIO = 0.40
EDGE_MARGIN_PX     = 40
MIN_OUT_FRAMES     = 4
MIN_SPEED_PX       = 0.8
ASSIGN_MAX_DIST    = 260
PREBUFFER_SECONDS  = 10
BACKUP_GRACE_SECONDS = 3
BACKUP_GATE_SCALE    = 1.2
DRAW_WINDOW_STRIP  = True
DRAW_TRASH_TRAJ    = True
TRAJ_TAIL          = 20

VIDEO_EXTS = ('.mp4', '.mov', '.avi', '.mkv', '.wmv', '.m4v')

# =========================================================
# 유틸 함수 (첫 코드 동일)
# =========================================================
def bbox_center(x1, y1, x2, y2):
    return (0.5 * (x1 + x2), 0.5 * (y1 + y2))

def point_in_bbox(pt, box, margin=0):
    x1, y1, x2, y2 = box
    return (x1 - margin <= pt[0] <= x2 + margin) and (y1 - margin <= pt[1] <= y2 + margin)

def top_window_strip(box, ratio=WINDOW_STRIP_RATIO):
    x1, y1, x2, y2 = box
    h = max(0, y2 - y1)
    return (x1, y1, x2, y1 + int(h * ratio))

def l2_dist(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

# =========================================================
# 모델 1회 로드 + GPU 옵션
# =========================================================
model = RTDETR(YOLO_WEIGHTS)

# 라벨 검증
print("Model names:", model.names)
assert CAR_LABEL in model.names.values(), f"'{CAR_LABEL}' 라벨이 모델에 없습니다."
for n in TRASH_LABELS:
    assert n in model.names.values(), f"'{n}' 라벨이 모델에 없습니다."

if DEVICE.startswith("cuda"):
    model.to("cuda")
    if USE_TORCH_COMPILE:
        try:
            model.model = torch.compile(model.model, mode="max-autotune")
            print("torch.compile enabled.")
        except Exception as e:
            print(f"torch.compile 사용 불가: {e}")

PREDICT_KW = dict(device=DEVICE, half=USE_HALF, verbose=False)
if IMG_SIZE is not None:
    PREDICT_KW["imgsz"] = IMG_SIZE

# 더미 예열(초기 지연 완화)
try:
    dummy = np.zeros((IMG_SIZE or 640, IMG_SIZE or 640, 3), dtype=np.uint8)
    _ = model.predict(dummy, conf=0.01, **PREDICT_KW)
except Exception:
    pass

# =========================================================
# 트래커 팩토리 (영상마다 새로 생성)
# =========================================================
def make_car_tracker():
    return DeepSort(max_age=30)

def make_trash_tracker():
    # 1) mobilenet 임베더 (GPU)
    try:
        return DeepSort(
            max_age=20, n_init=2, nn_budget=50,
            max_iou_distance=0.9, max_cosine_distance=0.7,
            embedder="mobilenet", half=True, bgr=True, embedder_gpu=True
        )
    except TypeError:
        pass
    # 2) torchreid
    try:
        return DeepSort(
            max_age=20, n_init=2, nn_budget=50,
            max_iou_distance=0.9, max_cosine_distance=0.7,
            embedder="torchreid", half=True, bgr=True, embedder_gpu=True
        )
    except TypeError:
        pass
    # 3) 구버전 최소 인자
    try:
        return DeepSort(max_age=20, n_init=2, max_iou_distance=0.9)
    except TypeError:
        pass
    return DeepSort(max_age=20)

# =========================================================
# 단일 영상 처리 (첫 코드 로직 100% 동일)
# =========================================================
def process_one_video(video_path):
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    output_dir = os.path.join(OUTPUT_ROOT, video_name)
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ 영상 열기 실패: {video_name} ({video_path})")
        return

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)  or 1280)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 720)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    print(f"\n✅ 영상 불러옴: {video_name} ({frame_count}프레임, {fps:.2f}FPS, {width}x{height})")

    # --- 트래커 & 상태 초기화 (영상마다 새로) ---
    car_tracker = make_car_tracker()
    trash_tracker = make_trash_tracker()

    prebuffer_len = max(1, int(round(fps * PREBUFFER_SECONDS)))
    prebuffer_annotated = deque(maxlen=prebuffer_len)

    trash_states = {}
    trash_traj   = {}
    car_scores   = {}

    littering_detected = False
    tracked_car_id     = None
    recording_started  = False
    clip_frames        = []

    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1

        # ---------- YOLO 추론 ----------
        results = model.predict(frame, conf=CONF_THRES, **PREDICT_KW)[0]

        car_dets, trash_dets = [], []
        yolo_detections = []

        for box in results.boxes:
            cls_id = int(box.cls.item())
            label = model.names[cls_id]
            conf  = float(box.conf.item())
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            if conf < CONF_THRES:
                continue
            if label == CAR_LABEL:
                car_dets.append(([x1, y1, (x2 - x1), (y2 - y1)], conf, 'car'))
            if label in TRASH_LABELS:
                trash_dets.append(([x1, y1, (x2 - x1), (y2 - y1)], conf, 'trash'))
            yolo_detections.append({'label': label, 'conf': conf, 'bbox': (x1, y1, x2, y2)})

        # ---------- 트래킹 ----------
        car_tracks   = car_tracker.update_tracks(car_dets,   frame=frame)
        trash_tracks = trash_tracker.update_tracks(trash_dets, frame=frame)

        # ---------- 차량 bbox dict ----------
        cars = {}
        for ct in car_tracks:
            if not ct.is_confirmed() or ct.track_id is None:
                continue
            l, t, w, h = map(int, ct.to_ltwh())
            cars[ct.track_id] = (l, t, l + w, t + h)
            car_scores.setdefault(ct.track_id, 0)

        # ---------- 시각화 프레임 ----------
        draw_frame = frame.copy()

        for car_id, (x1, y1, x2, y2) in cars.items():
            color = COLORS['littering_car'] if car_id == tracked_car_id else COLORS['car']
            cv2.rectangle(draw_frame, (x1, y1), (x2, y2), color, 2)
            label_text = f"LITTERING CAR ID {car_id}" if car_id == tracked_car_id else f"Car ID {car_id}"
            cv2.putText(draw_frame, label_text, (x1, max(0, y1 - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            if DRAW_WINDOW_STRIP:
                wx1, wy1, wx2, wy2 = top_window_strip((x1, y1, x2, y2))
                cv2.rectangle(draw_frame, (wx1, wy1), (wx2, wy2), COLORS['window_strip'], 1)

        for det in yolo_detections:
            if det['label'] == CAR_LABEL:
                continue
            x1, y1, x2, y2 = det['bbox']
            color = COLORS.get(det['label'], (255, 255, 255))
            cv2.rectangle(draw_frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(draw_frame, f"{det['label']} {det['conf']:.2f}", (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        # ---------- 쓰레기 트랙 상태 갱신 ----------
        for tt in trash_tracks:
            if not tt.is_confirmed() or tt.track_id is None:
                continue

            tl, ttop, tw, th = map(int, tt.to_ltwh())
            tb = (tl, ttop, tl + tw, ttop + th)
            tc = bbox_center(*tb)

            traj = trash_traj.setdefault(tt.track_id, deque(maxlen=TRAJ_TAIL))
            traj.append(tc)

            st = trash_states.setdefault(tt.track_id, {
                'first_pos': tc,
                'first_frame': frame_idx,
                'came_from_car_id': None,
                'has_crossed': False,
                'out_frames': 0,
                'last_pos': tc,
                'total_dist': 0.0,
                'steps': 0,
                'resolved': False
            })

            # 최초 출현: 차 내부/경계 근접 + 창문 보너스
            if st['came_from_car_id'] is None:
                candidate, best_cost = None, 1e9
                for car_id, cb in cars.items():
                    near_or_inside = point_in_bbox(tc, cb, margin=EDGE_MARGIN_PX)
                    if not near_or_inside:
                        continue
                    wx1, wy1, wx2, wy2 = top_window_strip(cb)
                    window_bonus = 0 if point_in_bbox(tc, (wx1, wy1, wx2, wy2)) else 20
                    c = l2_dist(tc, bbox_center(*cb)) + window_bonus
                    if c < best_cost:
                        best_cost, candidate = c, car_id
                if candidate is not None:
                    st['came_from_car_id'] = candidate

            # 가장자리 출원 허용
            if st['came_from_car_id'] is None:
                for car_id, cb in cars.items():
                    x1, y1, x2, y2 = cb
                    margin = int(EDGE_MARGIN_PX * 1.5)
                    near_edge = point_in_bbox(tc, (x1 - margin, y1 - margin, x2 + margin, y2 + margin)) \
                                and not point_in_bbox(tc, cb, margin=0)
                    if near_edge:
                        st['came_from_car_id'] = car_id
                        break

            # 이동/속도 누적
            d = l2_dist(tc, st['last_pos'])
            st['total_dist'] += d
            st['steps'] += 1
            st['last_pos'] = tc

            # 경계 이탈 검사
            cid = st['came_from_car_id']
            if cid is not None:
                car_box = cars.get(cid)
                if car_box is not None:
                    inside_now = point_in_bbox(tc, car_box, margin=EDGE_MARGIN_PX)
                    if not inside_now and not st['has_crossed']:
                        st['has_crossed'] = True
                        st['out_frames'] = 1
                    elif st['has_crossed']:
                        st['out_frames'] += 1

            # 사건 성립 판정
            if (not st['resolved']) and st['has_crossed'] and st['out_frames'] >= MIN_OUT_FRAMES and st['steps'] > 0:
                avg_speed = st['total_dist'] / max(1, st['steps'])
                if avg_speed >= MIN_SPEED_PX:
                    final_car = st['came_from_car_id']
                    if final_car in cars:
                        ccenter = bbox_center(*cars[final_car])
                        if l2_dist(tc, ccenter) <= ASSIGN_MAX_DIST:
                            car_scores[final_car] += 1
                            if not littering_detected:
                                littering_detected = True
                                tracked_car_id = final_car
                                recording_started = True
                                # 프리버퍼 추가
                                clip_frames.extend(list(prebuffer_annotated))
                                print(f"🚨 무단투기 차량 감지! ID: {tracked_car_id} @ Frame {frame_idx}")
                            st['resolved'] = True

            # 시각화
            if DRAW_TRASH_TRAJ and len(traj) >= 2:
                pts = [(int(p[0]), int(p[1])) for p in traj]
                for i in range(1, len(pts)):
                    cv2.line(draw_frame, pts[i - 1], pts[i], COLORS['trash_traj'], 2)

            cv2.rectangle(draw_frame, (tl, ttop), (tl + tw, ttop + th), COLORS['trash'], 2)
            cv2.putText(draw_frame, f"trash TID {tt.track_id}", (tl, max(0, ttop - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLORS['trash'], 2)

        # 점수 표시
        if car_scores:
            top_id = max(car_scores, key=lambda k: car_scores[k])
            score_text = f"Top Car ID: {top_id}  Score: {car_scores[top_id]}"
            cv2.putText(draw_frame, score_text, (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, COLORS['text'], 2)

        # ✅ 프리버퍼 저장
        prebuffer_annotated.append(draw_frame.copy())

        # ✅ 녹화 중이면 현재 프레임 추가 (빈 업데이트 호출 없음)
        if recording_started:
            clip_frames.append(draw_frame)
            # 차량 ID가 화면에서 사라지면 종료
            active_ids = [t.track_id for t in car_tracks if t.is_confirmed()]
            if tracked_car_id not in active_ids:
                print(f"✅ 무단투기 차량 ID {tracked_car_id} 추적 종료 @ Frame {frame_idx}")
                break

        # ---------- 백업 휴리스틱 ----------
        if not littering_detected and cars and trash_states:
            grace = int(max(1, fps * BACKUP_GRACE_SECONDS))
            pending = [
                (tid, st) for tid, st in trash_states.items()
                if (not st['resolved']) and (frame_idx - st['first_frame'] >= grace)
            ]
            if pending:
                pending.sort(key=lambda kv: (-kv[1]['out_frames'], -kv[1]['first_frame']))
                tid, st = pending[0]
                tc_now = st['last_pos']
                best, bestd = None, 1e9
                for car_id, cb in cars.items():
                    d = l2_dist(tc_now, bbox_center(*cb))
                    if d < bestd:
                        bestd, best = d, car_id
                if best is not None and bestd <= ASSIGN_MAX_DIST * BACKUP_GATE_SCALE:
                    car_scores[best] += 1
                    littering_detected = True
                    tracked_car_id = best
                    recording_started = True
                    clip_frames.extend(list(prebuffer_annotated))
                    st['resolved'] = True
                    print(f"⚠️ 백업 휴리스틱 발동: 차량 ID {tracked_car_id} (거리 {bestd:.1f}) @ Frame {frame_idx}")

    cap.release()

    # --- 클립 저장 ---
    if clip_frames:
        output_clip_path = os.path.join(output_dir, f"{video_name}_littering_clip2.mp4")
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out_fps = fps if fps and fps > 0 else 30.0
        out = cv2.VideoWriter(output_clip_path, fourcc, out_fps, (width, height))
        for f in clip_frames:
            out.write(f)
        out.release()
        print(f"🎬 무단투기 클립 저장 완료: {output_clip_path}")
    else:
        print("⚠️ 무단투기 사건이 성립하지 않아 클립이 생성되지 않았습니다.")

# =========================================================
# 폴더 순회 메인
# =========================================================
def main():
    if not os.path.isdir(VIDEO_DIR):
        print(f"❌ 폴더가 존재하지 않습니다: {VIDEO_DIR}")
        return

    os.makedirs(OUTPUT_ROOT, exist_ok=True)

    video_files = []
    for name in os.listdir(VIDEO_DIR):
        if name.lower().endswith(VIDEO_EXTS):
            video_files.append(os.path.join(VIDEO_DIR, name))

    video_files.sort()
    if not video_files:
        print(f"⚠️ 처리할 영상이 없습니다. (확장자: {VIDEO_EXTS})")
        return

    print(f"📂 총 {len(video_files)}개 영상 처리 시작:")
    for i, vp in enumerate(video_files, 1):
        print(f"\n[{i}/{len(video_files)}] ▶ {os.path.basename(vp)}")
        try:
            process_one_video(vp)
        except Exception as e:
            print(f"❗ 처리 중 오류 발생: {os.path.basename(vp)} -> {e}")

    print("\n✅ 전체 처리 완료!")

if __name__ == "__main__":
    main()


In [ ]:
!mkdir -p /content/drive/MyDrive/exports
!cp -r littering_clips_943_ours /content/drive/MyDrive/exports/

In [ ]:
# -*- coding: utf-8 -*-
"""
여러 영상 일괄 처리 — 단일 영상 로직 100% 유지 + GPU 가속
- 모델은 1회 로드(GPU/FP16/torch.compile 옵션)
- 각 영상마다 트래커/상태 초기화
- 첫 번째 코드의 휴리스틱/파라미터/시각화 완전 동일
"""

import os
import cv2
import math
import numpy as np
from collections import deque

import torch
from ultralytics import RTDETR
from deep_sort_realtime.deepsort_tracker import DeepSort

# =========================================================
# 런타임 / 장치 설정 (GPU 자동 사용)
# =========================================================
try:
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass
except Exception:
    pass

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
USE_HALF = (DEVICE.startswith("cuda"))
IMG_SIZE = None            # 예: 960 지정 가능. None이면 모델 기본값 사용
USE_TORCH_COMPILE = True   # 환경에 따라 자동 실패-무시

# =========================================================
# 설정부 (첫 코드와 동일)
# =========================================================
YOLO_WEIGHTS = '/content/best_detr.pt'  # ✅ 첫 번째 코드와 동일 경로 사용

TRASH_LABELS = ['trash']
CAR_LABEL = 'car'

VIDEO_DIR = '/content/drive/MyDrive/video_ITS'   # ✅ 폴더 경로
OUTPUT_ROOT = 'littering_clips_943_ITS'                 # ✅ 출력 루트

COLORS = {
    'car': (255, 0, 0),
    'littering_car': (0, 0, 255),
    'trash': (255, 0, 255),
    'text': (220, 220, 220),
    'window_strip': (180, 180, 0),
    'trash_traj': (200, 120, 255),
    'human': (0, 255, 255),
    'hand': (255, 255, 255),
    'license_plate': (0, 255, 0)
}

# =========================================================
# 파라미터 (첫 코드 값 유지)
# =========================================================
CONF_THRES = 0.6
WINDOW_STRIP_RATIO = 0.40
EDGE_MARGIN_PX     = 40
MIN_OUT_FRAMES     = 4
MIN_SPEED_PX       = 0.8
ASSIGN_MAX_DIST    = 260
PREBUFFER_SECONDS  = 10
BACKUP_GRACE_SECONDS = 3
BACKUP_GATE_SCALE    = 1.2
DRAW_WINDOW_STRIP  = True
DRAW_TRASH_TRAJ    = True
TRAJ_TAIL          = 20

VIDEO_EXTS = ('.mp4', '.mov', '.avi', '.mkv', '.wmv', '.m4v')

# =========================================================
# 유틸 함수 (첫 코드 동일)
# =========================================================
def bbox_center(x1, y1, x2, y2):
    return (0.5 * (x1 + x2), 0.5 * (y1 + y2))

def point_in_bbox(pt, box, margin=0):
    x1, y1, x2, y2 = box
    return (x1 - margin <= pt[0] <= x2 + margin) and (y1 - margin <= pt[1] <= y2 + margin)

def top_window_strip(box, ratio=WINDOW_STRIP_RATIO):
    x1, y1, x2, y2 = box
    h = max(0, y2 - y1)
    return (x1, y1, x2, y1 + int(h * ratio))

def l2_dist(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

# =========================================================
# 모델 1회 로드 + GPU 옵션
# =========================================================
model = RTDETR(YOLO_WEIGHTS)

# 라벨 검증
print("Model names:", model.names)
assert CAR_LABEL in model.names.values(), f"'{CAR_LABEL}' 라벨이 모델에 없습니다."
for n in TRASH_LABELS:
    assert n in model.names.values(), f"'{n}' 라벨이 모델에 없습니다."

if DEVICE.startswith("cuda"):
    model.to("cuda")
    if USE_TORCH_COMPILE:
        try:
            model.model = torch.compile(model.model, mode="max-autotune")
            print("torch.compile enabled.")
        except Exception as e:
            print(f"torch.compile 사용 불가: {e}")

PREDICT_KW = dict(device=DEVICE, half=USE_HALF, verbose=False)
if IMG_SIZE is not None:
    PREDICT_KW["imgsz"] = IMG_SIZE

# 더미 예열(초기 지연 완화)
try:
    dummy = np.zeros((IMG_SIZE or 640, IMG_SIZE or 640, 3), dtype=np.uint8)
    _ = model.predict(dummy, conf=0.01, **PREDICT_KW)
except Exception:
    pass

# =========================================================
# 트래커 팩토리 (영상마다 새로 생성)
# =========================================================
def make_car_tracker():
    return DeepSort(max_age=30)

def make_trash_tracker():
    # 1) mobilenet 임베더 (GPU)
    try:
        return DeepSort(
            max_age=20, n_init=2, nn_budget=50,
            max_iou_distance=0.9, max_cosine_distance=0.7,
            embedder="mobilenet", half=True, bgr=True, embedder_gpu=True
        )
    except TypeError:
        pass
    # 2) torchreid
    try:
        return DeepSort(
            max_age=20, n_init=2, nn_budget=50,
            max_iou_distance=0.9, max_cosine_distance=0.7,
            embedder="torchreid", half=True, bgr=True, embedder_gpu=True
        )
    except TypeError:
        pass
    # 3) 구버전 최소 인자
    try:
        return DeepSort(max_age=20, n_init=2, max_iou_distance=0.9)
    except TypeError:
        pass
    return DeepSort(max_age=20)

# =========================================================
# 단일 영상 처리 (첫 코드 로직 100% 동일)
# =========================================================
def process_one_video(video_path):
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    output_dir = os.path.join(OUTPUT_ROOT, video_name)
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ 영상 열기 실패: {video_name} ({video_path})")
        return

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)  or 1280)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 720)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    print(f"\n✅ 영상 불러옴: {video_name} ({frame_count}프레임, {fps:.2f}FPS, {width}x{height})")

    # --- 트래커 & 상태 초기화 (영상마다 새로) ---
    car_tracker = make_car_tracker()
    trash_tracker = make_trash_tracker()

    prebuffer_len = max(1, int(round(fps * PREBUFFER_SECONDS)))
    prebuffer_annotated = deque(maxlen=prebuffer_len)

    trash_states = {}
    trash_traj   = {}
    car_scores   = {}

    littering_detected = False
    tracked_car_id     = None
    recording_started  = False
    clip_frames        = []

    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1

        # ---------- YOLO 추론 ----------
        results = model.predict(frame, conf=CONF_THRES, **PREDICT_KW)[0]

        car_dets, trash_dets = [], []
        yolo_detections = []

        for box in results.boxes:
            cls_id = int(box.cls.item())
            label = model.names[cls_id]
            conf  = float(box.conf.item())
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            if conf < CONF_THRES:
                continue
            if label == CAR_LABEL:
                car_dets.append(([x1, y1, (x2 - x1), (y2 - y1)], conf, 'car'))
            if label in TRASH_LABELS:
                trash_dets.append(([x1, y1, (x2 - x1), (y2 - y1)], conf, 'trash'))
            yolo_detections.append({'label': label, 'conf': conf, 'bbox': (x1, y1, x2, y2)})

        # ---------- 트래킹 ----------
        car_tracks   = car_tracker.update_tracks(car_dets,   frame=frame)
        trash_tracks = trash_tracker.update_tracks(trash_dets, frame=frame)

        # ---------- 차량 bbox dict ----------
        cars = {}
        for ct in car_tracks:
            if not ct.is_confirmed() or ct.track_id is None:
                continue
            l, t, w, h = map(int, ct.to_ltwh())
            cars[ct.track_id] = (l, t, l + w, t + h)
            car_scores.setdefault(ct.track_id, 0)

        # ---------- 시각화 프레임 ----------
        draw_frame = frame.copy()

        for car_id, (x1, y1, x2, y2) in cars.items():
            color = COLORS['littering_car'] if car_id == tracked_car_id else COLORS['car']
            cv2.rectangle(draw_frame, (x1, y1), (x2, y2), color, 2)
            label_text = f"LITTERING CAR ID {car_id}" if car_id == tracked_car_id else f"Car ID {car_id}"
            cv2.putText(draw_frame, label_text, (x1, max(0, y1 - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            if DRAW_WINDOW_STRIP:
                wx1, wy1, wx2, wy2 = top_window_strip((x1, y1, x2, y2))
                cv2.rectangle(draw_frame, (wx1, wy1), (wx2, wy2), COLORS['window_strip'], 1)

        for det in yolo_detections:
            if det['label'] == CAR_LABEL:
                continue
            x1, y1, x2, y2 = det['bbox']
            color = COLORS.get(det['label'], (255, 255, 255))
            cv2.rectangle(draw_frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(draw_frame, f"{det['label']} {det['conf']:.2f}", (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        # ---------- 쓰레기 트랙 상태 갱신 ----------
        for tt in trash_tracks:
            if not tt.is_confirmed() or tt.track_id is None:
                continue

            tl, ttop, tw, th = map(int, tt.to_ltwh())
            tb = (tl, ttop, tl + tw, ttop + th)
            tc = bbox_center(*tb)

            traj = trash_traj.setdefault(tt.track_id, deque(maxlen=TRAJ_TAIL))
            traj.append(tc)

            st = trash_states.setdefault(tt.track_id, {
                'first_pos': tc,
                'first_frame': frame_idx,
                'came_from_car_id': None,
                'has_crossed': False,
                'out_frames': 0,
                'last_pos': tc,
                'total_dist': 0.0,
                'steps': 0,
                'resolved': False
            })

            # 최초 출현: 차 내부/경계 근접 + 창문 보너스
            if st['came_from_car_id'] is None:
                candidate, best_cost = None, 1e9
                for car_id, cb in cars.items():
                    near_or_inside = point_in_bbox(tc, cb, margin=EDGE_MARGIN_PX)
                    if not near_or_inside:
                        continue
                    wx1, wy1, wx2, wy2 = top_window_strip(cb)
                    window_bonus = 0 if point_in_bbox(tc, (wx1, wy1, wx2, wy2)) else 20
                    c = l2_dist(tc, bbox_center(*cb)) + window_bonus
                    if c < best_cost:
                        best_cost, candidate = c, car_id
                if candidate is not None:
                    st['came_from_car_id'] = candidate

            # 가장자리 출원 허용
            if st['came_from_car_id'] is None:
                for car_id, cb in cars.items():
                    x1, y1, x2, y2 = cb
                    margin = int(EDGE_MARGIN_PX * 1.5)
                    near_edge = point_in_bbox(tc, (x1 - margin, y1 - margin, x2 + margin, y2 + margin)) \
                                and not point_in_bbox(tc, cb, margin=0)
                    if near_edge:
                        st['came_from_car_id'] = car_id
                        break

            # 이동/속도 누적
            d = l2_dist(tc, st['last_pos'])
            st['total_dist'] += d
            st['steps'] += 1
            st['last_pos'] = tc

            # 경계 이탈 검사
            cid = st['came_from_car_id']
            if cid is not None:
                car_box = cars.get(cid)
                if car_box is not None:
                    inside_now = point_in_bbox(tc, car_box, margin=EDGE_MARGIN_PX)
                    if not inside_now and not st['has_crossed']:
                        st['has_crossed'] = True
                        st['out_frames'] = 1
                    elif st['has_crossed']:
                        st['out_frames'] += 1

            # 사건 성립 판정
            if (not st['resolved']) and st['has_crossed'] and st['out_frames'] >= MIN_OUT_FRAMES and st['steps'] > 0:
                avg_speed = st['total_dist'] / max(1, st['steps'])
                if avg_speed >= MIN_SPEED_PX:
                    final_car = st['came_from_car_id']
                    if final_car in cars:
                        ccenter = bbox_center(*cars[final_car])
                        if l2_dist(tc, ccenter) <= ASSIGN_MAX_DIST:
                            car_scores[final_car] += 1
                            if not littering_detected:
                                littering_detected = True
                                tracked_car_id = final_car
                                recording_started = True
                                # 프리버퍼 추가
                                clip_frames.extend(list(prebuffer_annotated))
                                print(f"🚨 무단투기 차량 감지! ID: {tracked_car_id} @ Frame {frame_idx}")
                            st['resolved'] = True

            # 시각화
            if DRAW_TRASH_TRAJ and len(traj) >= 2:
                pts = [(int(p[0]), int(p[1])) for p in traj]
                for i in range(1, len(pts)):
                    cv2.line(draw_frame, pts[i - 1], pts[i], COLORS['trash_traj'], 2)

            cv2.rectangle(draw_frame, (tl, ttop), (tl + tw, ttop + th), COLORS['trash'], 2)
            cv2.putText(draw_frame, f"trash TID {tt.track_id}", (tl, max(0, ttop - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLORS['trash'], 2)

        # 점수 표시
        if car_scores:
            top_id = max(car_scores, key=lambda k: car_scores[k])
            score_text = f"Top Car ID: {top_id}  Score: {car_scores[top_id]}"
            cv2.putText(draw_frame, score_text, (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, COLORS['text'], 2)

        # ✅ 프리버퍼 저장
        prebuffer_annotated.append(draw_frame.copy())

        # ✅ 녹화 중이면 현재 프레임 추가 (빈 업데이트 호출 없음)
        if recording_started:
            clip_frames.append(draw_frame)
            # 차량 ID가 화면에서 사라지면 종료
            active_ids = [t.track_id for t in car_tracks if t.is_confirmed()]
            if tracked_car_id not in active_ids:
                print(f"✅ 무단투기 차량 ID {tracked_car_id} 추적 종료 @ Frame {frame_idx}")
                break

        # ---------- 백업 휴리스틱 ----------
        if not littering_detected and cars and trash_states:
            grace = int(max(1, fps * BACKUP_GRACE_SECONDS))
            pending = [
                (tid, st) for tid, st in trash_states.items()
                if (not st['resolved']) and (frame_idx - st['first_frame'] >= grace)
            ]
            if pending:
                pending.sort(key=lambda kv: (-kv[1]['out_frames'], -kv[1]['first_frame']))
                tid, st = pending[0]
                tc_now = st['last_pos']
                best, bestd = None, 1e9
                for car_id, cb in cars.items():
                    d = l2_dist(tc_now, bbox_center(*cb))
                    if d < bestd:
                        bestd, best = d, car_id
                if best is not None and bestd <= ASSIGN_MAX_DIST * BACKUP_GATE_SCALE:
                    car_scores[best] += 1
                    littering_detected = True
                    tracked_car_id = best
                    recording_started = True
                    clip_frames.extend(list(prebuffer_annotated))
                    st['resolved'] = True
                    print(f"⚠️ 백업 휴리스틱 발동: 차량 ID {tracked_car_id} (거리 {bestd:.1f}) @ Frame {frame_idx}")

    cap.release()

    # --- 클립 저장 ---
    if clip_frames:
        output_clip_path = os.path.join(output_dir, f"{video_name}_littering_clip2.mp4")
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out_fps = fps if fps and fps > 0 else 30.0
        out = cv2.VideoWriter(output_clip_path, fourcc, out_fps, (width, height))
        for f in clip_frames:
            out.write(f)
        out.release()
        print(f"🎬 무단투기 클립 저장 완료: {output_clip_path}")
    else:
        print("⚠️ 무단투기 사건이 성립하지 않아 클립이 생성되지 않았습니다.")

# =========================================================
# 폴더 순회 메인
# =========================================================
def main():
    if not os.path.isdir(VIDEO_DIR):
        print(f"❌ 폴더가 존재하지 않습니다: {VIDEO_DIR}")
        return

    os.makedirs(OUTPUT_ROOT, exist_ok=True)

    video_files = []
    for name in os.listdir(VIDEO_DIR):
        if name.lower().endswith(VIDEO_EXTS):
            video_files.append(os.path.join(VIDEO_DIR, name))

    video_files.sort()
    if not video_files:
        print(f"⚠️ 처리할 영상이 없습니다. (확장자: {VIDEO_EXTS})")
        return

    print(f"📂 총 {len(video_files)}개 영상 처리 시작:")
    for i, vp in enumerate(video_files, 1):
        print(f"\n[{i}/{len(video_files)}] ▶ {os.path.basename(vp)}")
        try:
            process_one_video(vp)
        except Exception as e:
            print(f"❗ 처리 중 오류 발생: {os.path.basename(vp)} -> {e}")

    print("\n✅ 전체 처리 완료!")

if __name__ == "__main__":
    main()


In [ ]:
!mkdir -p /content/drive/MyDrive/exports
!cp -r littering_clips_943_ITS /content/drive/MyDrive/exports/